# 02 - Preprocessing & Validation
===

Data cleaning, validation, and preprocessing pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'figure.figsize': (10, 6)})
sns.set_style('whitegrid')

DATA_DIR = Path('./data/processed')
OUTPUT_DIR = Path('./outputs/figures/preprocessing')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Raw Data

In [ ]:
raw_path = DATA_DIR / 'ais_raw.csv'
if raw_path.exists():
    raw = pd.read_csv(raw_path, nrows=100000)
    print(f"Raw records: {len(raw):,}")
    print(f"Raw columns: {list(raw.columns)}")
else:
    print("Raw data not found, using processed data")
    raw = pd.read_parquet(DATA_DIR / 'ais_features.parquet')
    print(f"Using processed: {len(raw):,}")

## Coordinate Validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

lat = raw['LAT'].dropna()
sns.histplot(lat, bins=50, ax=axes[0], color='teal')
axes[0].axvline(90, color='red', linestyle='--', alpha=0.5, label='Invalid (91°)')
axes[0].axvline(-90, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Latitude')
axes[0].set_title('Latitude Distribution')
axes[0].legend()

lon = raw['LON'].dropna()
sns.histplot(lon, bins=50, ax=axes[1], color='coral')
axes[1].axvline(180, color='red', linestyle='--', alpha=0.5, label='Invalid (181°)')
axes[1].axvline(-180, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Longitude')
axes[1].set_title('Longitude Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coordinate_validation.png', dpi=300, bbox_inches='tight')
plt.show()

## Speed Validation

In [ ]:
sog = raw['SOG'].dropna()
invalid_sog = (sog >= 102.3).sum()
implausible = (sog > 50).sum()

print(f"Total Speed records: {len(sog):,}")
print(f"Invalid (>102.3): {invalid_sog:,} ({invalid_sog/len(sog)*100:.2f}%)")
print(f"Implausible (>50): {implausible:,} ({implausible/len(sog)*100:.2f}%)")

## MMSI Validation

In [ ]:
mmsi = raw['MMSI']
valid_mmsi = mmsi.between(200000000, 799999999)
special_mmsi = mmsi[mmsi.between(990000000, 999999999)]

print(f"Valid civilian MMSI: {valid_mmsi.sum():,} ({valid_mmsi.mean()*100:.1f}%)")
print(f"Special (ATON): {len(special_mmsi):,}")

## Temporal Validation

In [ ]:
df['BaseDateTime'] = pd.to_datetime(df['BaseDateTime'], errors='coerce')
now = pd.Timestamp.now()
cutoff = pd.Timestamp('2010-01-01')

future = (df['BaseDateTime'] > now).sum()
old = (df['BaseDateTime'] < cutoff).sum()
missing = df['BaseDateTime'].isna().sum()

print(f"Future dates: {future:,}")
print(f"Pre-2010: {old:,}")
print(f"Missing: {missing:,}")

## Duplicate Detection

In [ ]:
dup_cols = ['MMSI', 'BaseDateTime'] if 'BaseDateTime' in raw.columns else ['MMSI']
n_duplicates = raw.duplicated(subset=dup_cols).sum()
print(f"Duplicates: {n_duplicates:,} ({n_duplicates/len(raw)*100:.2f}%)")

## Data Quality Summary

In [ ]:
# Compute quality metrics
quality = {
    'total_records': len(raw),
    'valid_lat': ((raw['LAT'] >= -90) & (raw['LAT'] <= 90)).sum() / len(raw) * 100,
    'valid_lon': ((raw['LON'] >= -180) & (raw['LON'] <= 180)).sum() / len(raw) * 100,
    'valid_sog': (raw['SOG'] < 102.3).sum() / len(raw) * 100,
    'valid_mmsi': raw['MMSI'].between(200000000, 799999999).sum() / len(raw) * 100,
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(quality.keys()), list(quality.values()), color='steelblue')
ax.set_xlabel('Valid %')
ax.set_title('Data Quality Metrics')
ax.set_xlim(0, 100)
for bar, val in zip(bars, quality.values()):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'data_quality.png', dpi=300, bbox_inches='tight')
plt.show()

## Preprocessing Report

In [ ]:
print("="*50)
print("PREPROCESSING VALIDATION COMPLETE")
print("="*50)
print(f"Raw data: {len(raw):,} records")
print(f"Features: {len(df.columns)} columns")
print(f"Output: {OUTPUT_DIR}")